# 08 - Trayectorias de Optuna y aprendizaje CNN

Este nodo es **sólo de lectura**: reconstruye el historial persistido de los nodos 06, 07, 09 y 10 sin abrir los NIfTI ni interferir con un entrenamiento en curso. Lee:

- las bases SQLite de Optuna mediante una copia transaccional temporal;
- los `training_history.csv` / `history.csv` guardados después de cada época;
- los scores de cada fold incluidos en los trials completos;
- las evaluaciones finales de cinco folds cuando estén disponibles.

Por eso los resultados sobreviven a un apagado: están en disco, no únicamente en memoria RAM. Para ver avances nuevos basta con volver a ejecutar este notebook.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import FileLink, Markdown, clear_output, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modeling.trajectory_viz import (
    acquisition_family_figure,
    best_fold_comparison_figure,
    calibration_figure,
    diagnostic_figure,
    discover_runs,
    efficiency_figure,
    final_fold_figure,
    final_metrics_figure,
    final_progress_figure,
    finalist_agreement_figure,
    hyperparameter_figure,
    learning_curve_figure,
    load_trajectory_snapshot,
    network_comparison_figure,
    optimization_figure,
    probability_distribution_figure,
    write_dashboard,
)
from modeling.trajectory_viz.figures import PLOTLY_CONFIG

# Opcional: "node06:cnn_compact_v1,node07:dat_spect_slab_v4,node09:node09_fine_v1,node10:node10_hybrid_v1".
RUN_KEYS_TEXT = os.environ.get("DAT_TRAJECTORY_RUNS", "").strip()
RUN_KEYS = [value.strip() for value in RUN_KEYS_TEXT.split(",") if value.strip()] or None
EXPORT_DIR = (
    PROJECT_ROOT / "outputs" / "private_eda" / "trajectory_dashboard" / "latest"
)


## 1. Descubrimiento y snapshot consistente

Si `RUN_KEYS` queda en `None`, se toma automáticamente el run con actividad más reciente de cada nodo. La copia temporal de SQLite incluye incluso trials `RUNNING`, `PRUNED` o `FAIL`; el archivo original permanece intacto.


In [2]:
catalog = discover_runs(PROJECT_ROOT)
display(catalog)

snapshot = load_trajectory_snapshot(PROJECT_ROOT, RUN_KEYS)
display(Markdown(f"**Snapshot UTC:** `{snapshot.created_at}`"))
display(snapshot.catalog)
if snapshot.warnings:
    display(Markdown("**Advertencias de lectura:**\n\n" + "\n".join(f"- {item}" for item in snapshot.warnings)))

coverage = pd.DataFrame({
    "indicador": [
        "trials registrados",
        "trials completos",
        "trials en curso",
        "trials podados",
        "trials fallidos",
        "archivos de historial",
        "filas época-fold",
        "trayectorias diagnosticables",
        "finalistas CV5 observados",
        "modelos-fold finales completos",
        "modelos-fold finales en curso",
        "predicciones OOF finales disponibles",
    ],
    "valor": [
        len(snapshot.trials),
        int(snapshot.trials.get("state", pd.Series(dtype=str)).eq("COMPLETE").sum()),
        int(snapshot.trials.get("state", pd.Series(dtype=str)).eq("RUNNING").sum()),
        int(snapshot.trials.get("state", pd.Series(dtype=str)).eq("PRUNED").sum()),
        int(snapshot.trials.get("state", pd.Series(dtype=str)).eq("FAIL").sum()),
        int(snapshot.catalog.get("history_files", pd.Series(dtype=int)).sum()),
        len(snapshot.histories),
        int(snapshot.diagnostics.get("validation_points", pd.Series(dtype=int)).ge(4).sum()),
        int(snapshot.final_progress.get("candidate_id", pd.Series(dtype=str)).nunique()),
        int(snapshot.final_progress.get("status", pd.Series(dtype=str)).eq("complete").sum()),
        int(snapshot.final_progress.get("status", pd.Series(dtype=str)).eq("training").sum()),
        len(snapshot.final_predictions),
    ],
})
display(coverage)


,run_key,node,run_id,run_dir,has_optuna_database,history_files,latest_activity_utc
0,node06:cnn_compact_v1,node06,cnn_compact_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,True,277,2026-08-29 15:04:41.712899446+00:00
1,node07:dat_spect_slab_v4,node07,dat_spect_slab_v4,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,True,275,2026-08-31 04:04:26.972816944+00:00
2,node07:dat_spect_slab_v3,node07,dat_spect_slab_v3,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,False,0,NaT
3,node07:dat_spect_slab_v2,node07,dat_spect_slab_v2,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,False,0,NaT
4,node07:dat_spect_slab_v1,node07,dat_spect_slab_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,False,0,NaT
5,node09:node09_fine_v1,node09,node09_fine_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,True,187,2026-09-02 00:16:26.556841850+00:00
6,node10:node10_hybrid_v1,node10,node10_hybrid_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,False,0,NaT


**Snapshot UTC:** `2026-09-02T03:04:20.222918+00:00`

,run_key,node,run_id,run_dir,has_optuna_database,history_files,latest_activity_utc,trials_complete,trials_running,trials_pruned,trials_fail,trials_waiting
0,node06:cnn_compact_v1,node06,cnn_compact_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,True,277,2026-08-29 15:04:41.712899446+00:00,76,0,23,4,0
1,node07:dat_spect_slab_v4,node07,dat_spect_slab_v4,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,True,275,2026-08-31 04:04:26.972816944+00:00,76,0,19,1,0
2,node09:node09_fine_v1,node09,node09_fine_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,True,187,2026-09-02 00:16:26.556841850+00:00,54,0,5,0,0
3,node10:node10_hybrid_v1,node10,node10_hybrid_v1,C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT...,False,0,NaT,0,0,0,0,0


,indicador,valor
0,trials registrados,258
1,trials completos,206
2,trials en curso,0
3,trials podados,47
4,trials fallidos,5
5,archivos de historial,739
6,filas época-fold,10738
7,trayectorias diagnosticables,694
8,finalistas CV5 observados,9
9,modelos-fold finales completos,45


## 2. Trayectoria global de la búsqueda

Los puntos son trials completos; la línea discontinua es el mejor log loss acumulado dentro de cada estudio. Una meseta indica que ampliar ciegamente el número de trials probablemente rinde poco; mejoras tardías indican que todavía vale la pena explorar alrededor de esa zona.


In [3]:
fig_optimization = optimization_figure(snapshot.trials)
fig_optimization.show(config=PLOTLY_CONFIG)


## 3. Comparación entre redes y estabilidad entre folds

La primera figura compara toda la distribución de trials. La segunda compara únicamente el mejor trial actual de cada red y conserva sus tres folds separados. El promedio aislado puede esconder un fold frágil; por eso ambos gráficos son necesarios.


In [4]:
fig_networks = network_comparison_figure(snapshot.trials)
fig_networks.show(config=PLOTLY_CONFIG)

fig_folds = best_fold_comparison_figure(snapshot.trials, snapshot.fold_scores)
fig_folds.show(config=PLOTLY_CONFIG)

fig_efficiency = efficiency_figure(snapshot.trials)
fig_efficiency.show(config=PLOTLY_CONFIG)


## 4. Evaluación final de cinco folds

Esta sección se actualiza mientras corre `run_final_stage`. Distingue resultados **parciales** de métricas finales: un candidato sólo recibe calibración cruzada cuando completó sus cinco folds. La matriz muestra el avance y el log loss de cada fold; la dispersión permite detectar si una media aparentemente buena depende de un único fold favorable.


In [5]:
fig_final_progress = final_progress_figure(snapshot.final_progress)
fig_final_progress.show(config=PLOTLY_CONFIG)

fig_final_folds = final_fold_figure(snapshot.final_progress)
fig_final_folds.show(config=PLOTLY_CONFIG)

fig_final_metrics = final_metrics_figure(snapshot.final_metrics)
fig_final_metrics.show(config=PLOTLY_CONFIG)

if not snapshot.final_metrics.empty:
    final_metric_columns = [
        column for column in [
            "candidate_id", "architecture", "feature_variant", "lateral_strategy",
            "folds_completed", "expected_folds", "n_predictions", "metrics_source",
            "raw_log_loss", "calibrated_log_loss", "raw_auc", "calibrated_auc",
            "raw_brier", "calibrated_brier", "raw_ece_10", "calibrated_ece_10",
        ] if column in snapshot.final_metrics
    ]
    display(snapshot.final_metrics[final_metric_columns].sort_values(
        ["folds_completed", "raw_log_loss"], ascending=[False, True]
    ))


,candidate_id,architecture,feature_variant,lateral_strategy,folds_completed,expected_folds,n_predictions,metrics_source,raw_log_loss,calibrated_log_loss,raw_auc,calibrated_auc,raw_brier,calibrated_brier,raw_ece_10,calibrated_ece_10
6,555b81376733bb67,slab2d,image_radiomics_sbr,random_flip,5,5,1362,official_final_metrics,0.547940,0.543655,0.801811,0.800133,0.182645,0.181365,0.050513,0.037625
3,23f042f62f40d03e,slab2d,image_radiomics_sbr,random_flip,5,5,1362,official_final_metrics,0.556924,0.551498,0.799906,0.796822,0.187166,0.184972,0.064860,0.045810
5,fb93ea204b0ec32f,2.5d,image_radiomics_sbr,random_flip,5,5,1362,official_final_metrics,0.560599,0.552242,0.797649,0.793100,0.187826,0.184982,0.071564,0.043068
7,c56be99202976455,2.5d,image_radiomics_sbr,random_flip,5,5,1362,official_final_metrics,0.561650,0.558443,0.785729,0.784103,0.188128,0.187208,0.044513,0.042201
0,c5f41331c124f406,2.5d,image_radiomics,not_applicable,5,5,1362,official_final_metrics,0.564639,0.565744,0.779125,0.776339,0.191864,0.192020,0.045336,0.037593
2,43765c3f64a985b1,2.5d,image_only,not_applicable,5,5,1362,official_final_metrics,0.567424,0.560384,0.785089,0.783389,0.192720,0.189813,0.058971,0.021902
1,4f0001840e93bffa,3d,image_radiomics,not_applicable,5,5,1362,official_final_metrics,0.581195,0.585177,0.766753,0.760606,0.197633,0.199116,0.052351,0.049702
4,3f9bbf24164bd1dc,3d,image_radiomics_sbr,random_flip,5,5,1362,official_final_metrics,0.586020,0.565622,0.780836,0.775836,0.195744,0.190079,0.087526,0.029703
8,a37f424ba911f6cc,3d,image_radiomics_sbr,random_flip,5,5,1362,official_final_metrics,0.610864,0.571392,0.775264,0.772863,0.202112,0.193613,0.087669,0.031732


### Separación, calibración y complementariedad

El selector usa las predicciones OOF disponibles del finalista. Para candidatos incompletos la lectura es provisional y sólo se muestra la probabilidad sin calibrar. El panel de acuerdo ayuda a decidir si un ensemble puede aportar: redes idénticas en sus probabilidades agregan poco, mientras diferencias razonables pueden ser complementarias.


In [6]:
final_candidate_rows = (
    snapshot.final_metrics.sort_values(
        ["folds_completed", "raw_log_loss"], ascending=[False, True]
    )
    if not snapshot.final_metrics.empty
    else pd.DataFrame()
)
final_candidate_options = []
for row in final_candidate_rows.itertuples(index=False):
    calibrated = getattr(row, "calibrated_log_loss", float("nan"))
    score = calibrated if pd.notna(calibrated) else row.raw_log_loss
    status = "final" if bool(row.evaluation_complete) else "parcial"
    final_candidate_options.append((
        f"{row.architecture} · {str(row.candidate_id)[:8]} · "
        f"{row.folds_completed}/{row.expected_folds} folds · {status} · log loss={score:.4f}",
        str(row.candidate_id),
    ))

final_candidate_selector = widgets.Dropdown(
    options=final_candidate_options,
    description="Finalista:",
    layout=widgets.Layout(width="95%"),
)
final_candidate_output = widgets.Output()

def render_final_candidate(change=None):
    with final_candidate_output:
        clear_output(wait=True)
        if not final_candidate_options:
            display(Markdown("Aún no hay predicciones OOF finales."))
            return
        candidate_id = final_candidate_selector.value
        probability_distribution_figure(
            snapshot.final_predictions, candidate_id
        ).show(config=PLOTLY_CONFIG)
        calibration_figure(
            snapshot.final_predictions, candidate_id
        ).show(config=PLOTLY_CONFIG)

final_candidate_selector.observe(render_final_candidate, names="value")
display(final_candidate_selector, final_candidate_output)
render_final_candidate()

fig_final_agreement = finalist_agreement_figure(snapshot.final_predictions)
fig_final_agreement.show(config=PLOTLY_CONFIG)

fig_final_families = acquisition_family_figure(snapshot.final_predictions)
fig_final_families.show(config=PLOTLY_CONFIG)


Dropdown(description='Finalista:', layout=Layout(width='95%'), options=(('slab2d · 555b8137 · 5/5 folds · fina…

Output()

## 5. Espacio de hiperparámetros

Cada línea es un trial. El color representa log loss: permite ver interacciones que una tabla de números oculta. El selector separa los estudios para no comparar parámetros con significados incompatibles.


In [ ]:
study_rows = (
    snapshot.trials[["run_key", "study_name"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["run_key", "study_name"])
    if not snapshot.trials.empty
    else pd.DataFrame()
)
study_options = [
    (f"{row.run_key} · {row.study_name}", (row.run_key, row.study_name))
    for row in study_rows.itertuples(index=False)
]
study_selector = widgets.Dropdown(
    options=study_options,
    description="Estudio:",
    layout=widgets.Layout(width="95%"),
)
study_output = widgets.Output()

def render_hyperparameters(change=None):
    with study_output:
        clear_output(wait=True)
        if not study_options:
            display(Markdown("Aún no hay estudios Optuna para visualizar."))
            return
        run_key, study_name = study_selector.value
        hyperparameter_figure(
            snapshot.trials, study_name, run_key=run_key
        ).show(config=PLOTLY_CONFIG)

study_selector.observe(render_hyperparameters, names="value")
display(study_selector, study_output)
render_hyperparameters()


Dropdown(description='Estudio:', layout=Layout(width='95%'), options=(('node06:cnn_compact_v1 · cnn_25d_image_…

Output()

## 6. Curvas por época y fold

Se muestran por separado el objetivo de entrenamiento y el log loss de validación. No deben restarse literalmente: el objetivo `train` incluye regularización por consistencia y, en los nodos 07, 09 y 10, puede incluir objetivos auxiliares. Sí sirven sus tendencias: entrenamiento descendente junto con validación que rebota es señal de posible sobreajuste.


In [8]:
trajectory_rows = (
    snapshot.histories[
        ["run_key", "phase", "study_name", "trajectory_id", "trial_number", "objective_log_loss"]
    ]
    .drop_duplicates()
    .sort_values(["phase", "objective_log_loss", "run_key", "study_name"], na_position="last")
    if not snapshot.histories.empty
    else pd.DataFrame()
)
trajectory_options = []
for row in trajectory_rows.itertuples(index=False):
    trial = f"trial {int(row.trial_number)}" if pd.notna(row.trial_number) else f"hash {str(row.trajectory_id)[:8]}"
    score = f" · {row.objective_log_loss:.4f}" if pd.notna(row.objective_log_loss) else " · activo/no mapeado"
    trajectory_options.append((
        f"{row.run_key} · {row.phase} · {row.study_name} · {trial}{score}",
        (row.run_key, row.phase, str(row.trajectory_id)),
    ))

trajectory_selector = widgets.Dropdown(
    options=trajectory_options,
    description="Trayectoria:",
    layout=widgets.Layout(width="95%"),
)
trajectory_output = widgets.Output()

def render_learning_curve(change=None):
    with trajectory_output:
        clear_output(wait=True)
        if not trajectory_options:
            display(Markdown("Aún no hay historiales por época."))
            return
        run_key, phase, trajectory_id = trajectory_selector.value
        learning_curve_figure(
            snapshot.histories,
            trajectory_id,
            run_key=run_key,
            phase=phase,
        ).show(config=PLOTLY_CONFIG)

trajectory_selector.observe(render_learning_curve, names="value")
display(trajectory_selector, trajectory_output)
render_learning_curve()


Dropdown(description='Trayectoria:', layout=Layout(width='95%'), options=(('node09:node09_fine_v1 · final_cv5 …

Output()

## 7. Señales de sobreajuste y subajuste

El diagnóstico es deliberadamente heurístico. **Sobreajuste** requiere que el objetivo de entrenamiento siga bajando mientras validación se aleja al menos `0.03` de su mejor punto. **Posible subajuste/optimización débil** exige poco aprendizaje en train y validación cercana o peor que `0.67`. Los cinco folds finales sólo validan en la última época por diseño, por lo que no alcanzan para este diagnóstico temporal.


In [9]:
fig_diagnostics = diagnostic_figure(snapshot.diagnostics)
fig_diagnostics.show(config=PLOTLY_CONFIG)

if not snapshot.trajectory_summary.empty:
    display(
        snapshot.trajectory_summary[
            [
                "run_key", "phase", "study_name", "trial_number", "folds_observed",
                "mean_best_validation", "std_best_validation", "mean_rebound",
                "median_best_epoch", "overfit_fraction", "underfit_fraction",
                "unstable_fraction",
            ]
        ].head(40)
    )


,run_key,phase,study_name,trial_number,folds_observed,mean_best_validation,std_best_validation,mean_rebound,median_best_epoch,overfit_fraction,underfit_fraction,unstable_fraction
0,node09:node09_fine_v1,final_cv5,node09_slab2d_23f042f62f,9.0,5,0.547909,0.047728,0.000000,25.0,0.000000,0.0,0.0
1,node07:dat_spect_slab_v4,final_cv5,node07_slab2d_random_flip_image_radiomics_sbr,0.0,5,0.556890,0.053239,0.000000,9.0,0.000000,0.0,0.0
2,node07:dat_spect_slab_v4,final_cv5,node07_25d_random_flip_image_radiomics_sbr,0.0,5,0.560563,0.067719,0.000000,9.0,0.000000,0.0,0.0
3,node09:node09_fine_v1,final_cv5,node09_25d_fb93ea204b,15.0,5,0.561621,0.041441,0.000000,15.0,0.000000,0.0,0.0
4,node06:cnn_compact_v1,final_cv5,cnn_25d_image_radiomics,11.0,5,0.572887,0.031557,0.000000,7.0,0.000000,0.0,0.0
5,node06:cnn_compact_v1,final_cv5,cnn_25d_image_only,22.0,5,0.573165,0.046298,0.000000,7.0,0.000000,0.0,0.0
6,node06:cnn_compact_v1,final_cv5,cnn_3d_image_radiomics,12.0,5,0.577384,0.044820,0.000000,3.0,0.000000,0.0,0.0
7,node07:dat_spect_slab_v4,final_cv5,node07_3d_random_flip_image_radiomics_sbr,0.0,5,0.585977,0.079528,0.000000,3.0,0.000000,0.0,0.0
8,node09:node09_fine_v1,final_cv5,node09_3d_3f9bbf2416,15.0,5,0.610805,0.081202,0.000000,18.0,0.000000,0.0,0.0
9,node09:node09_fine_v1,search_cv3,node09_slab2d_23f042f62f,9.0,3,0.524060,0.024492,0.004302,25.0,0.000000,0.0,0.0


## 8. Exportación persistente

Se guarda un HTML interactivo autocontenido y snapshots CSV. Volver a ejecutar esta celda actualiza `latest` de forma atómica; no borra ni altera artefactos de los nodos 06, 07, 09 y 10.


In [10]:
dashboard_path = write_dashboard(
    snapshot,
    EXPORT_DIR,
    {
        "Avance final CV5": fig_final_progress,
        "Métricas OOF finales": fig_final_metrics,
        "Estabilidad final por fold": fig_final_folds,
        "Distribución OOF": probability_distribution_figure(snapshot.final_predictions),
        "Calibración OOF": calibration_figure(snapshot.final_predictions),
        "Acuerdo entre finalistas": fig_final_agreement,
        "Robustez por adquisición": fig_final_families,
        "Trayectoria de Optuna": fig_optimization,
        "Comparación entre redes": fig_networks,
        "Costo frente a rendimiento": fig_efficiency,
        "Mejores trials por fold": fig_folds,
        "Espacio de hiperparámetros": hyperparameter_figure(snapshot.trials),
        "Curvas de aprendizaje": learning_curve_figure(snapshot.histories),
        "Diagnóstico de ajuste": fig_diagnostics,
    },
)
display(Markdown(f"**Dashboard actualizado:** `{dashboard_path}`"))
display(FileLink(dashboard_path))


**Dashboard actualizado:** `C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\outputs\private_eda\trajectory_dashboard\latest\dashboard.html`

C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\outputs\private_eda\trajectory_dashboard\latest\dashboard.html